# Data preparation

## Load libraries

In [0]:
# Import warnings
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Load dataset

### Audit the files

In [0]:
import glob
import os

# Path to the data
DATA_PATH = "/Volumes/nyc_taxi_analytics/raw/landing_zone/trip_records/"

# Find all parquet files
files = sorted(glob.glob(os.path.join(DATA_PATH, "*.parquet")))

# Audit
print(f"Total files found: {len(files)}\n")
for f in files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"{os.path.basename(f):50s} {size_mb:8.1f} MB")

### Sample and Load

In [0]:
import pandas as pd
import numpy as np
import glob
import os

DATA_PATH = "/Volumes/nyc_taxi_analytics/raw/landing_zone/trip_records/"

# Fix sort order - by month number
files = sorted(
    glob.glob(os.path.join(DATA_PATH, "*.parquet")),
    key=lambda x: int(os.path.basename(x).split("-")[1].replace(".parquet", ""))
)

sampled_months = []

for f in files:
    month_name = os.path.basename(f).replace(".parquet", "")
    print(f"Loading {month_name}...", end=" ")

    # Load full month
    df_month = pd.read_parquet(f)

    # Extract date and hour for stratification
    df_month["pickup_date"] = df_month["tpep_pickup_datetime"].dt.date
    df_month["pickup_hour"] = df_month["tpep_pickup_datetime"].dt.hour

    # Stratified 5% sample — sparse groups get at least 1 row
    df_sampled = (
        df_month
        .groupby(["pickup_date", "pickup_hour"], group_keys=False)
        .apply(lambda grp: 
               grp.sample(frac=0.05, random_state=42) if len(grp) >= 20 
               else grp.sample(n=1, random_state=42)
        )
    )

    sampled_months.append(df_sampled)
    print(f"→ {len(df_sampled):,} rows sampled from {len(df_month):,} ({len(df_sampled)/len(df_month)*100:.1f}%)")

    # Free memory immediately
    del df_month

# Combine all months
df = pd.concat(sampled_months, ignore_index=True)

# Drop helper columns used only for sampling
df.drop(columns=["pickup_date", "pickup_hour"], inplace=True)

print(f"\n✅ Final dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## Schema Inspection

In [0]:
print("=" * 55)
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("=" * 55)

# Column names, data types and memory usage
schema = pd.DataFrame({
    "dtype"        : df.dtypes,
    "memory_MB"    : df.memory_usage(deep=True)[1:] / 1024**2,
    "null_count"   : df.isnull().sum(),
    "null_percent" : (df.isnull().sum() / len(df) * 100).round(2)
})

print(schema.to_string())
print("=" * 55)
print(f"Total Memory: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

## Data inspection

In [0]:
print("=" * 55)
print("FIRST 5 ROWS")
print("=" * 55)
df.head()

In [0]:
print("\n")
print("=" * 55)
print("LAST 5 ROWS")
print("=" * 55)
df.tail()

In [0]:
print("\n")
print("=" * 55)
print("STATISTICAL SUMMARY")
print("=" * 55)
pd.set_option("display.float_format", "{:.2f}".format)
df.describe(include="all")

# Data cleaning

In [0]:
# Always work on a copy — preserve raw data
df_clean = df.copy()
rows_before = len(df_clean)

# Filter only 2023 trips
df_clean = df_clean[
    (df_clean["tpep_pickup_datetime"].dt.year == 2023) &
    (df_clean["tpep_dropoff_datetime"].dt.year == 2023)
]

rows_after = len(df_clean)
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")

In [0]:
rows_before = len(df_clean)

# Columns that must never be negative
non_negative_cols = [
    "fare_amount", "extra", "mta_tax", "tip_amount",
    "tolls_amount", "improvement_surcharge",
    "total_amount", "congestion_surcharge"
]

# Build a mask — keep only rows where ALL these columns are >= 0
mask = (df_clean[non_negative_cols] >= 0).all(axis=1)
df_clean = df_clean[mask]

rows_after = len(df_clean)
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")

In [0]:
rows_before = len(df_clean)

# Valid passenger count: 1 to 6 (NYC cab max capacity)
# Also drop nulls in passenger_count
df_clean = df_clean[
    df_clean["passenger_count"].notna() &
    df_clean["passenger_count"].between(1, 6)
]

rows_after = len(df_clean)
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")

In [0]:
rows_before = len(df_clean)

# Valid RatecodeIDs per TLC data dictionary: 1 through 6
valid_rate_codes = [1, 2, 3, 4, 5, 6]

df_clean = df_clean[
    df_clean["RatecodeID"].notna() &
    df_clean["RatecodeID"].isin(valid_rate_codes)
]

rows_after = len(df_clean)
print(f"Rows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")

In [0]:
# rows_before = len(df_clean)

# # Calculate IQR for trip_distance
# Q1 = df_clean["trip_distance"].quantile(0.25)
# Q3 = df_clean["trip_distance"].quantile(0.75)
# IQR = Q3 - Q1
# upper_fence = Q3 + 1.5 * IQR

# print(f"Q1            : {Q1:.2f} miles")
# print(f"Q3            : {Q3:.2f} miles")
# print(f"IQR           : {IQR:.2f} miles")
# print(f"Upper fence   : {upper_fence:.2f} miles")

# # Remove only upper outliers
# df_clean = df_clean[df_clean["trip_distance"] <= upper_fence]

# rows_after = len(df_clean)
# print(f"\nRows before : {rows_before:,}")
# print(f"Rows after  : {rows_after:,}")
# print(f"Rows removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")

## Revised: Domain-Knowledge Distance Filter

In [0]:
rows_before = len(df_clean)

# Domain knowledge based distance filter:
# - Zero distance only valid for RatecodeID 5 (negotiated fare)
# - Upper cap at 100 miles — beyond this is a data error
df_clean = df_clean[
    (
        (df_clean["trip_distance"] == 0) & 
        (df_clean["RatecodeID"] == 5)
    ) |
    (
        (df_clean["trip_distance"] > 0) & 
        (df_clean["trip_distance"] <= 100)
    )
]

rows_after = len(df_clean)
print(f"Q1 trip distance  : {df_clean['trip_distance'].quantile(0.25):.2f} miles")
print(f"Q3 trip distance  : {df_clean['trip_distance'].quantile(0.75):.2f} miles")
print(f"Max trip distance : {df_clean['trip_distance'].max():.2f} miles")
print(f"\nRows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")

In [0]:
rows_before = len(df_clean)

# Domain knowledge fare filter:
# Minimum NYC meter drop: $3.00
# Maximum realistic fare: $300
df_clean = df_clean[
    df_clean["fare_amount"].between(3.0, 300.0)
]

rows_after = len(df_clean)
print(f"Min fare amount : ${df_clean['fare_amount'].min():.2f}")
print(f"Max fare amount : ${df_clean['fare_amount'].max():.2f}")
print(f"Mean fare amount: ${df_clean['fare_amount'].mean():.2f}")
print(f"\nRows before : {rows_before:,}")
print(f"Rows after  : {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")

In [0]:
rows_before = len(df_clean)

# Coalesce: take airport_fee if not null, otherwise take Airport_fee
df_clean["airport_fee_clean"] = df_clean["airport_fee"].combine_first(df_clean["Airport_fee"])

# Drop both original columns
df_clean.drop(columns=["airport_fee", "Airport_fee"], inplace=True)

# Verify the merge
print(f"Null count in merged column : {df_clean['airport_fee_clean'].isnull().sum():,}")
print(f"Unique values               : {sorted(df_clean['airport_fee_clean'].dropna().unique())}")
print(f"Value counts:")
print(df_clean["airport_fee_clean"].value_counts().head(10))
print(f"\nRows before : {rows_before:,}")
print(f"Rows after  : {len(df_clean):,}")

In [0]:
# Before — check current memory
mem_before = df_clean.memory_usage(deep=True).sum() / 1024**2
print(f"Memory before type fixes: {mem_before:.1f} MB\n")

# 1. Categorical columns
cat_cols = ["VendorID", "RatecodeID", "payment_type", 
            "store_and_fwd_flag", "PULocationID", "DOLocationID"]
for col in cat_cols:
    df_clean[col] = df_clean[col].astype("category")

# 2. Passenger count — nullable integer
df_clean["passenger_count"] = df_clean["passenger_count"].astype("Int8")

# 3. Airport fee — float32 is sufficient (values are 0, 1.25, 1.75)
df_clean["airport_fee_clean"] = df_clean["airport_fee_clean"].astype("float32")

# After — check new memory
mem_after = df_clean.memory_usage(deep=True).sum() / 1024**2
print(f"Memory after type fixes : {mem_after:.1f} MB")
print(f"Memory saved            : {mem_before - mem_after:.1f} MB ({(mem_before - mem_after)/mem_before*100:.1f}%)")

# Verify dtypes
print(f"\nUpdated dtypes:")
print(df_clean.dtypes)

In [0]:
print("=" * 55)
print("       DATA CLEANING FINAL REPORT")
print("=" * 55)

print(f"\n{'SHAPE':}")
print(f"  Raw dataset    : {df.shape[0]:>10,} rows × {df.shape[1]} columns")
print(f"  Clean dataset  : {df_clean.shape[0]:>10,} rows × {df_clean.shape[1]} columns")
print(f"  Rows removed   : {df.shape[0] - df_clean.shape[0]:>10,} ({(df.shape[0] - df_clean.shape[0])/df.shape[0]*100:.2f}%)")
print(f"  Rows retained  : {df_clean.shape[0]/df.shape[0]*100:.2f}%")

print(f"\n{'MEMORY':}")
print(f"  Raw dataset    : {df.memory_usage(deep=True).sum()/1024**2:>8.1f} MB")
print(f"  Clean dataset  : {df_clean.memory_usage(deep=True).sum()/1024**2:>8.1f} MB")
print(f"  Memory saved   : {(df.memory_usage(deep=True).sum() - df_clean.memory_usage(deep=True).sum())/1024**2:>8.1f} MB")

print(f"\n{'NULL CHECK':}")
null_counts = df_clean.isnull().sum()
if null_counts.sum() == 0:
    print("  ✅ Zero nulls remaining across all columns")
else:
    print(null_counts[null_counts > 0])

print(f"\n{'COLUMN CHANGES':}")
print(f"  Raw columns    : {list(df.columns)}")
print(f"  Clean columns  : {list(df_clean.columns)}")

print(f"\n{'DATE RANGE':}")
print(f"  Earliest pickup: {df_clean['tpep_pickup_datetime'].min()}")
print(f"  Latest pickup  : {df_clean['tpep_pickup_datetime'].max()}")

print(f"\n{'KEY STATS':}")
print(f"  Fare  — min: ${df_clean['fare_amount'].min():.2f}  "
      f"mean: ${df_clean['fare_amount'].mean():.2f}  "
      f"max: ${df_clean['fare_amount'].max():.2f}")
print(f"  Dist  — min: {df_clean['trip_distance'].min():.2f}mi  "
      f"mean: {df_clean['trip_distance'].mean():.2f}mi  "
      f"max: {df_clean['trip_distance'].max():.2f}mi")
print(f"  Pass  — min: {df_clean['passenger_count'].min()}  "
      f"mean: {df_clean['passenger_count'].mean():.2f}  "
      f"max: {df_clean['passenger_count'].max()}")
print("=" * 55)

In [0]:
df_clean.to_parquet('/Volumes/nyc_taxi_analytics/raw/landing_zone/cleaned_copy/cleaned_nyc_taxi_data_ai.parquet')

# Feature engineering

## Temporal features

In [0]:
df_clean = pd.read_parquet('/Volumes/nyc_taxi_analytics/raw/landing_zone/cleaned_copy/cleaned_nyc_taxi_data_ai.parquet')

In [0]:
# Temporal features from pickup datetime
df_clean["pickup_hour"]        = df_clean["tpep_pickup_datetime"].dt.hour
df_clean["pickup_day_of_week"] = df_clean["tpep_pickup_datetime"].dt.day_name()
df_clean["pickup_month"]       = df_clean["tpep_pickup_datetime"].dt.month
df_clean["pickup_date"]        = df_clean["tpep_pickup_datetime"].dt.date
df_clean["is_weekend"]         = df_clean["tpep_pickup_datetime"].dt.dayofweek >= 5

# Trip duration in minutes
df_clean["trip_duration_mins"] = (
    (df_clean["tpep_dropoff_datetime"] - df_clean["tpep_pickup_datetime"])
    .dt.total_seconds() / 60
).round(2)

# Average speed in mph — avoid division by zero
df_clean["avg_speed_mph"] = np.where(
    df_clean["trip_duration_mins"] > 0,
    (df_clean["trip_distance"] / (df_clean["trip_duration_mins"] / 60)).round(2),
    np.nan
)

# Fare per mile — only meaningful for trips with distance > 0
df_clean["fare_per_mile"] = np.where(
    df_clean["trip_distance"] > 0,
    (df_clean["fare_amount"] / df_clean["trip_distance"]).round(2),
    np.nan
)

# Time of day buckets
def assign_time_of_day(hour):
    if 6 <= hour < 10:
        return "Morning Rush"
    elif 10 <= hour < 16:
        return "Midday"
    elif 16 <= hour < 20:
        return "Evening Rush"
    elif 20 <= hour < 24:
        return "Night"
    else:
        return "Late Night"

df_clean["time_of_day"] = df_clean["pickup_hour"].apply(assign_time_of_day)

# Verify
print(f"New shape     : {df_clean.shape}")
print(f"\nNew columns added:")
new_cols = ["pickup_hour", "pickup_day_of_week", "pickup_month","pickup_date", "is_weekend", "trip_duration_mins","avg_speed_mph", "fare_per_mile", "time_of_day"]
display(df_clean[new_cols].head(10))
print(f"\nTime of day distribution:")
print(df_clean["time_of_day"].value_counts())

# Univariate analysis

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Global plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 130

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle("Univariate Distributions — Distance, Fare & Duration", 
             fontsize=15, fontweight="bold", y=1.01)

cols = {
    "trip_distance"     : "Trip Distance (miles)",
    "fare_amount"       : "Fare Amount ($)",
    "trip_duration_mins": "Trip Duration (mins)"
}

for i, (col, label) in enumerate(cols.items()):
    
    data = df_clean[col].dropna()
    
    # --- Histogram + KDE ---
    ax1 = axes[i, 0]
    sns.histplot(data, bins=60, kde=True, ax=ax1, color="steelblue")
    ax1.set_title(f"{label} — Histogram + KDE")
    ax1.set_xlabel(label)
    ax1.set_ylabel("Count")
    ax1.axvline(data.mean(),   color="red",    linestyle="--", 
                linewidth=1.5, label=f"Mean: {data.mean():.2f}")
    ax1.axvline(data.median(), color="green",  linestyle="--", 
                linewidth=1.5, label=f"Median: {data.median():.2f}")
    ax1.legend(fontsize=8)

    # --- Box Plot ---
    ax2 = axes[i, 1]
    sns.boxplot(x=data, ax=ax2, color="lightsteelblue")
    ax2.set_title(f"{label} — Box Plot")
    ax2.set_xlabel(label)

plt.tight_layout()
# plt.savefig("univariate_distributions.png", bbox_inches="tight")
plt.show()
# print("Plot saved successfully")

In [0]:
rows_before = len(df_clean)

# Valid duration: at least 1 minute, at most 180 minutes (3 hours)
df_clean = df_clean[
    df_clean["trip_duration_mins"].between(1, 180)
]

rows_after = len(df_clean)
print(f"Min duration : {df_clean['trip_duration_mins'].min():.2f} mins")
print(f"Max duration : {df_clean['trip_duration_mins'].max():.2f} mins")
print(f"Mean duration: {df_clean['trip_duration_mins'].mean():.2f} mins")
print(f"\nRows before  : {rows_before:,}")
print(f"Rows after   : {rows_after:,}")
print(f"Rows removed : {rows_before - rows_after:,} ({(rows_before - rows_after)/rows_before*100:.2f}%)")

In [0]:
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
fig.suptitle("Univariate Distributions — Distance, Fare & Duration (Cleaned)", 
             fontsize=15, fontweight="bold", y=1.01)

cols = {
    "trip_distance"     : "Trip Distance (miles)",
    "fare_amount"       : "Fare Amount ($)",
    "trip_duration_mins": "Trip Duration (mins)"
}

for i, (col, label) in enumerate(cols.items()):
    
    data = df_clean[col].dropna()
    
    # --- Histogram + KDE ---
    ax1 = axes[i, 0]
    sns.histplot(data, bins=60, kde=True, ax=ax1, color="steelblue")
    ax1.set_title(f"{label} — Histogram + KDE")
    ax1.set_xlabel(label)
    ax1.set_ylabel("Count")
    ax1.axvline(data.mean(),   color="red",   linestyle="--",
                linewidth=1.5, label=f"Mean: {data.mean():.2f}")
    ax1.axvline(data.median(), color="green", linestyle="--",
                linewidth=1.5, label=f"Median: {data.median():.2f}")
    ax1.legend(fontsize=8)

    # --- Box Plot ---
    ax2 = axes[i, 1]
    sns.boxplot(x=data, ax=ax2, color="lightsteelblue")
    ax2.set_title(f"{label} — Box Plot")
    ax2.set_xlabel(label)

plt.tight_layout()
# plt.savefig("univariate_distributions_clean.png", bbox_inches="tight")
plt.show()
# print("Plot saved ✅")

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Univariate Distributions — Categorical Variables",
             fontsize=15, fontweight="bold")

# --- 1. Passenger Count ---
ax1 = axes[0, 0]
passenger_counts = df_clean["passenger_count"].value_counts().sort_index()
sns.barplot(x=passenger_counts.index.astype(str),
            y=passenger_counts.values, ax=ax1, color="steelblue")
ax1.set_title("Passenger Count Distribution")
ax1.set_xlabel("Number of Passengers")
ax1.set_ylabel("Trip Count")
for bar, val in zip(ax1.patches, passenger_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f"{val/len(df_clean)*100:.1f}%", ha="center", fontsize=8)

# --- 2. Payment Type ---
ax2 = axes[0, 1]
payment_map = {1: "Credit Card", 2: "Cash", 3: "No Charge", 4: "Dispute"}
payment_counts = df_clean["payment_type"].map(payment_map).value_counts()
sns.barplot(x=payment_counts.index, y=payment_counts.values,
            ax=ax2, palette="muted")
ax2.set_title("Payment Type Distribution")
ax2.set_xlabel("Payment Type")
ax2.set_ylabel("Trip Count")
for bar, val in zip(ax2.patches, payment_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f"{val/len(df_clean)*100:.1f}%", ha="center", fontsize=8)

# --- 3. Vendor Distribution ---
ax3 = axes[1, 0]
vendor_map = {1: "Creative Mobile\n(Vendor 1)", 2: "VeriFone\n(Vendor 2)"}
vendor_counts = df_clean["VendorID"].map(vendor_map).value_counts()
sns.barplot(x=vendor_counts.index, y=vendor_counts.values,
            ax=ax3, palette="muted")
ax3.set_title("Vendor Distribution")
ax3.set_xlabel("Vendor")
ax3.set_ylabel("Trip Count")
for bar, val in zip(ax3.patches, vendor_counts.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
             f"{val/len(df_clean)*100:.1f}%", ha="center", fontsize=8)

# --- 4. Rate Code Distribution ---
ax4 = axes[1, 1]
rate_map = {
    1: "Standard",
    2: "JFK",
    3: "Newark",
    4: "Nassau/West",
    5: "Negotiated",
    6: "Group Ride"
}
rate_counts = df_clean["RatecodeID"].map(rate_map).value_counts()
sns.barplot(x=rate_counts.index, y=rate_counts.values,
            ax=ax4, palette="muted")
ax4.set_title("Rate Code Distribution")
ax4.set_xlabel("Rate Code")
ax4.set_ylabel("Trip Count")
for bar, val in zip(ax4.patches, rate_counts.values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
             f"{val/len(df_clean)*100:.1f}%", ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("categorical_distributions.png", bbox_inches="tight")
plt.show()
print("Plot saved ✅")

# Financial EDA

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Revenue Composition Analysis", 
             fontsize=15, fontweight="bold")

# --- Revenue component averages ---
revenue_cols = {
    "fare_amount"          : "Base Fare",
    "tip_amount"           : "Tip",
    "tolls_amount"         : "Tolls",
    "congestion_surcharge" : "Congestion",
    "airport_fee_clean"    : "Airport Fee",
    "mta_tax"              : "MTA Tax",
    "improvement_surcharge": "Improvement",
    "extra"                : "Extra"
}

avg_components = {
    label: df_clean[col].mean() 
    for col, label in revenue_cols.items()
}
avg_df = pd.Series(avg_components).sort_values(ascending=False)

# --- Bar chart of average components ---
ax1 = axes[0]
bars = sns.barplot(x=avg_df.values, y=avg_df.index, 
                   ax=ax1, palette="Blues_r")
ax1.set_title("Average Revenue per Trip by Component")
ax1.set_xlabel("Average Amount ($)")
ax1.set_ylabel("Component")
for bar, val in zip(ax1.patches, avg_df.values):
    ax1.text(bar.get_width() + 0.1, 
             bar.get_y() + bar.get_height()/2,
             f"${val:.2f}", va="center", fontsize=9)

# --- Pie chart of revenue share ---
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    avg_df.values,
    labels=avg_df.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=sns.color_palette("Blues_r", len(avg_df))
)
ax2.set_title(f"Revenue Share\n(Avg Total: ${avg_df.sum():.2f} per trip)")

plt.tight_layout()
plt.savefig("revenue_composition.png", bbox_inches="tight")
plt.show()

# Print summary table
print("\nRevenue Component Summary:")
print("-" * 40)
total = avg_df.sum()
for name, val in avg_df.items():
    print(f"{name:<20} ${val:>6.2f}  ({val/total*100:.1f}%)")
print("-" * 40)
print(f"{'Total':<20} ${total:>6.2f}")

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle("Tip Behaviour Analysis", 
             fontsize=15, fontweight="bold")

# Credit card trips only — exclude cash tip bias
cc_trips = df_clean[df_clean["payment_type"].astype(str) == "1"].copy()
cc_trips["tip_rate"] = (cc_trips["tip_amount"] / 
                         cc_trips["fare_amount"] * 100).round(2)

print(f"Credit card trips        : {len(cc_trips):,}")
print(f"Average tip (all trips)  : ${df_clean['tip_amount'].mean():.2f}")
print(f"Average tip (card only)  : ${cc_trips['tip_amount'].mean():.2f}")
print(f"Average tip rate (card)  : {cc_trips['tip_rate'].mean():.1f}%")
print(f"Median tip rate (card)   : {cc_trips['tip_rate'].median():.1f}%")

# --- 1. Tip Amount Distribution (card only) ---
ax1 = axes[0, 0]
sns.histplot(cc_trips["tip_amount"], bins=50, kde=True,
             ax=ax1, color="steelblue")
ax1.set_title("Tip Amount Distribution\n(Credit Card Only)")
ax1.set_xlabel("Tip Amount ($)")
ax1.set_ylabel("Count")
ax1.axvline(cc_trips["tip_amount"].mean(), color="red",
            linestyle="--", linewidth=1.5,
            label=f"Mean: ${cc_trips['tip_amount'].mean():.2f}")
ax1.axvline(cc_trips["tip_amount"].median(), color="green",
            linestyle="--", linewidth=1.5,
            label=f"Median: ${cc_trips['tip_amount'].median():.2f}")
ax1.legend(fontsize=8)

# --- 2. Tip Rate Distribution ---
ax2 = axes[0, 1]
tip_rate_filtered = cc_trips[cc_trips["tip_rate"] <= 50]
sns.histplot(tip_rate_filtered["tip_rate"], bins=50, kde=True,
             ax=ax2, color="darkorange")
ax2.set_title("Tip Rate Distribution (% of Fare)\n(Credit Card, tip rate ≤ 50%)")
ax2.set_xlabel("Tip Rate (%)")
ax2.set_ylabel("Count")
ax2.axvline(tip_rate_filtered["tip_rate"].mean(), color="red",
            linestyle="--", linewidth=1.5,
            label=f"Mean: {tip_rate_filtered['tip_rate'].mean():.1f}%")
ax2.legend(fontsize=8)

# --- 3. Average Tip Rate by Time of Day ---
ax3 = axes[1, 0]
tod_order = ["Morning Rush", "Midday", "Evening Rush", "Night", "Late Night"]
tip_by_tod = (cc_trips.groupby("time_of_day")["tip_rate"]
              .mean()
              .reindex(tod_order))
sns.barplot(x=tip_by_tod.index, y=tip_by_tod.values,
            ax=ax3, palette="Blues_d")
ax3.set_title("Average Tip Rate by Time of Day\n(Credit Card Only)")
ax3.set_xlabel("Time of Day")
ax3.set_ylabel("Average Tip Rate (%)")
for bar, val in zip(ax3.patches, tip_by_tod.values):
    ax3.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.1,
             f"{val:.1f}%", ha="center", fontsize=9)

# --- 4. Average Tip Amount by Passenger Count ---
ax4 = axes[1, 1]
tip_by_pax = (cc_trips.groupby(
                  cc_trips["passenger_count"].astype(int))["tip_amount"]
              .mean())
sns.barplot(x=tip_by_pax.index.astype(str),
            y=tip_by_pax.values, ax=ax4, palette="Blues_d")
ax4.set_title("Average Tip Amount by Passenger Count\n(Credit Card Only)")
ax4.set_xlabel("Number of Passengers")
ax4.set_ylabel("Average Tip ($)")
for bar, val in zip(ax4.patches, tip_by_pax.values):
    ax4.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.02,
             f"${val:.2f}", ha="center", fontsize=9)

plt.tight_layout()
# plt.savefig("tip_behaviour.png", bbox_inches="tight")
plt.show()

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Fare Efficiency by Rate Code", 
             fontsize=15, fontweight="bold")

rate_map = {
    1: "Standard", 2: "JFK", 3: "Newark",
    4: "Nassau/West", 5: "Negotiated", 6: "Group Ride"
}

# Only trips with distance > 0 for fare_per_mile
# Only trips with duration > 0 for fare_per_minute
df_clean["fare_per_minute"] = np.where(
    df_clean["trip_duration_mins"] > 0,
    (df_clean["fare_amount"] / df_clean["trip_duration_mins"]).round(2),
    np.nan
)

df_eff = df_clean.copy()
df_eff["rate_label"] = df_eff["RatecodeID"].astype(float).map(rate_map)

# Filter valid rows for each metric
fpm_data  = df_eff[df_eff["fare_per_mile"].notna()]
fpmin_data = df_eff[df_eff["fare_per_minute"].notna()]

# Aggregate
fpm_avg   = (fpm_data.groupby("rate_label")["fare_per_mile"]
             .mean().sort_values(ascending=False))
fpmin_avg = (fpmin_data.groupby("rate_label")["fare_per_minute"]
             .mean().sort_values(ascending=False))

# --- Fare per Mile ---
ax1 = axes[0]
sns.barplot(x=fpm_avg.values, y=fpm_avg.index,
            ax=ax1, palette="Blues_r")
ax1.set_title("Average Fare per Mile by Rate Code")
ax1.set_xlabel("Fare per Mile ($/mile)")
ax1.set_ylabel("Rate Code")
for bar, val in zip(ax1.patches, fpm_avg.values):
    ax1.text(bar.get_width() + 0.1,
             bar.get_y() + bar.get_height()/2,
             f"${val:.2f}", va="center", fontsize=9)

# --- Fare per Minute ---
ax2 = axes[1]
sns.barplot(x=fpmin_avg.values, y=fpmin_avg.index,
            ax=ax2, palette="Blues_r")
ax2.set_title("Average Fare per Minute by Rate Code")
ax2.set_xlabel("Fare per Minute ($/min)")
ax2.set_ylabel("Rate Code")
for bar, val in zip(ax2.patches, fpmin_avg.values):
    ax2.text(bar.get_width() + 0.01,
             bar.get_y() + bar.get_height()/2,
             f"${val:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("fare_efficiency.png", bbox_inches="tight")
plt.show()

# Summary table
print("\nFare Efficiency Summary by Rate Code:")
print("-" * 55)
summary = df_eff.groupby("rate_label").agg(
    trips        = ("fare_amount", "count"),
    avg_fare     = ("fare_amount", "mean"),
    avg_distance = ("trip_distance", "mean"),
    avg_duration = ("trip_duration_mins", "mean"),
    avg_per_mile = ("fare_per_mile", "mean"),
    avg_per_min  = ("fare_per_minute", "mean")
).round(2)
print(summary.to_string())

# Temporal EDA

In [0]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Temporal Demand Patterns — NYC Yellow Taxi 2023",
             fontsize=15, fontweight="bold")

# Day order
day_order = ["Monday", "Tuesday", "Wednesday", 
             "Thursday", "Friday", "Saturday", "Sunday"]

# --- 1. Trip count by hour ---
ax1 = axes[0, 0]
hourly = df_clean.groupby("pickup_hour").size().reset_index(name="trips")
sns.barplot(x="pickup_hour", y="trips", data=hourly,
            ax=ax1, color="steelblue")
ax1.set_title("Trip Volume by Hour of Day")
ax1.set_xlabel("Hour of Day")
ax1.set_ylabel("Number of Trips")
ax1.set_xticks(range(0, 24, 2))

# --- 2. Trip count by day of week ---
ax2 = axes[0, 1]
daily = (df_clean.groupby("pickup_day_of_week")
         .size()
         .reindex(day_order)
         .reset_index(name="trips"))
sns.barplot(x="pickup_day_of_week", y="trips",
            data=daily, ax=ax2, palette="Blues_d")
ax2.set_title("Trip Volume by Day of Week")
ax2.set_xlabel("Day of Week")
ax2.set_ylabel("Number of Trips")
ax2.tick_params(axis="x", rotation=30)

# --- 3. Average fare by hour ---
ax3 = axes[1, 0]
fare_hourly = (df_clean.groupby("pickup_hour")["fare_amount"]
               .mean()
               .reset_index(name="avg_fare"))
sns.lineplot(x="pickup_hour", y="avg_fare",
             data=fare_hourly, ax=ax3,
             color="darkorange", linewidth=2.5, marker="o")
ax3.set_title("Average Fare by Hour of Day")
ax3.set_xlabel("Hour of Day")
ax3.set_ylabel("Average Fare ($)")
ax3.set_xticks(range(0, 24, 2))
ax3.fill_between(fare_hourly["pickup_hour"],
                 fare_hourly["avg_fare"],
                 alpha=0.15, color="darkorange")

# --- 4. Average duration by hour ---
ax4 = axes[1, 1]
dur_hourly = (df_clean.groupby("pickup_hour")["trip_duration_mins"]
              .mean()
              .reset_index(name="avg_duration"))
sns.lineplot(x="pickup_hour", y="avg_duration",
             data=dur_hourly, ax=ax4,
             color="green", linewidth=2.5, marker="o")
ax4.set_title("Average Trip Duration by Hour of Day")
ax4.set_xlabel("Hour of Day")
ax4.set_ylabel("Average Duration (mins)")
ax4.set_xticks(range(0, 24, 2))
ax4.fill_between(dur_hourly["pickup_hour"],
                 dur_hourly["avg_duration"],
                 alpha=0.15, color="green")

plt.tight_layout()
plt.savefig("temporal_demand.png", bbox_inches="tight")
plt.show()
print("Plot saved ✅")

In [0]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14))
fig.suptitle("Monthly Seasonality & Daily Time Series — 2023",
             fontsize=15, fontweight="bold")

# --- 1. Monthly trip volume ---
ax1 = axes[0]
monthly = (df_clean.groupby("pickup_month")
           .size()
           .reset_index(name="trips"))
monthly["month_name"] = monthly["pickup_month"].map({
    1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun",
    7:"Jul", 8:"Aug", 9:"Sep", 10:"Oct", 11:"Nov", 12:"Dec"
})
sns.barplot(x="month_name", y="trips", data=monthly,
            ax=ax1, palette="Blues_d")
ax1.set_title("Monthly Trip Volume — 2023")
ax1.set_xlabel("Month")
ax1.set_ylabel("Number of Trips")
for bar, row in zip(ax1.patches, monthly.itertuples()):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 500,
             f"{bar.get_height()/1000:.0f}K",
             ha="center", fontsize=8)

# --- 2. Monthly average fare ---
ax2 = axes[1]
monthly_fare = (df_clean.groupby("pickup_month")["fare_amount"]
                .mean()
                .reset_index(name="avg_fare"))
monthly_fare["month_name"] = monthly_fare["pickup_month"].map({
    1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun",
    7:"Jul", 8:"Aug", 9:"Sep", 10:"Oct", 11:"Nov", 12:"Dec"
})
sns.lineplot(x="month_name", y="avg_fare", data=monthly_fare,
             ax=ax2, color="darkorange", linewidth=2.5,
             marker="o", markersize=8)
ax2.set_title("Average Fare by Month — 2023")
ax2.set_xlabel("Month")
ax2.set_ylabel("Average Fare ($)")
ax2.fill_between(range(len(monthly_fare)),
                 monthly_fare["avg_fare"],
                 alpha=0.15, color="darkorange")
for i, row in monthly_fare.iterrows():
    ax2.text(i, row["avg_fare"] + 0.1,
             f"${row['avg_fare']:.2f}",
             ha="center", fontsize=8)

# --- 3. Daily trip volume time series ---
ax3 = axes[2]
daily_ts = (df_clean.groupby("pickup_date")
            .size()
            .reset_index(name="trips"))
daily_ts["pickup_date"] = pd.to_datetime(daily_ts["pickup_date"])
daily_ts["rolling_7d"]  = daily_ts["trips"].rolling(7).mean()

ax3.plot(daily_ts["pickup_date"], daily_ts["trips"],
         color="steelblue", alpha=0.4, linewidth=1,
         label="Daily trips")
ax3.plot(daily_ts["pickup_date"], daily_ts["rolling_7d"],
         color="darkblue", linewidth=2.5,
         label="7-day rolling avg")
ax3.set_title("Daily Trip Volume with 7-Day Rolling Average — 2023")
ax3.set_xlabel("Date")
ax3.set_ylabel("Number of Trips")
ax3.legend()

# Annotate key dips
dip_dates = {
    "New Year's\nDay"  : "2023-01-01",
    "Memorial\nDay"    : "2023-05-29",
    "July 4th"         : "2023-07-04",
    "Labor\nDay"       : "2023-09-04",
    "Thanksgiving"     : "2023-11-23",
    "Christmas"        : "2023-12-25"
}
for label, date in dip_dates.items():
    dt = pd.Timestamp(date)
    if dt in daily_ts["pickup_date"].values:
        y = daily_ts.loc[
            daily_ts["pickup_date"] == dt, "trips"].values[0]
        ax3.annotate(label,
                     xy=(dt, y),
                     xytext=(dt, y + 2000),
                     fontsize=7,
                     ha="center",
                     color="red",
                     arrowprops=dict(arrowstyle="->",
                                     color="red",
                                     lw=1))

plt.tight_layout()
plt.savefig("seasonality_timeseries.png", bbox_inches="tight")
plt.show()
print("Plot saved ✅")

# Bivariate and Multivariate analysis

In [0]:
fig, ax = plt.subplots(figsize=(14, 10))

# Select meaningful numeric columns only
numeric_cols = [
    "trip_distance", "fare_amount", "tip_amount",
    "tolls_amount", "total_amount", "trip_duration_mins",
    "avg_speed_mph", "fare_per_mile", "fare_per_minute",
    "passenger_count", "pickup_hour", "pickup_month",
    "congestion_surcharge", "extra"
]

corr_matrix = df_clean[numeric_cols].corr()

# Generate mask for upper triangle
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# Plot
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 8}
)

ax.set_title("Correlation Heatmap — NYC Yellow Taxi 2023",
             fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("correlation_heatmap.png", bbox_inches="tight")
plt.show()
print("Plot saved ✅")

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle("Hour × Day of Week Heatmaps — NYC Yellow Taxi 2023",
             fontsize=15, fontweight="bold")

day_order = ["Monday", "Tuesday", "Wednesday",
             "Thursday", "Friday", "Saturday", "Sunday"]

# --- Pivot 1: Trip Count ---
pivot_count = (df_clean
               .groupby(["pickup_day_of_week", "pickup_hour"])
               .size()
               .unstack(level=0)
               .reindex(columns=day_order))

sns.heatmap(
    pivot_count,
    ax=axes[0],
    cmap="YlOrRd",
    fmt=".0f",
    linewidths=0.3,
    cbar_kws={"label": "Trip Count"}
)
axes[0].set_title("Trip Volume\n(Hour × Day of Week)")
axes[0].set_xlabel("Day of Week")
axes[0].set_ylabel("Hour of Day")
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)

# --- Pivot 2: Average Fare ---
pivot_fare = (df_clean
              .groupby(["pickup_day_of_week", "pickup_hour"])["fare_amount"]
              .mean()
              .unstack(level=0)
              .reindex(columns=day_order))

sns.heatmap(
    pivot_fare,
    ax=axes[1],
    cmap="YlOrRd",
    fmt=".1f",
    annot=True,
    linewidths=0.3,
    annot_kws={"size": 6},
    cbar_kws={"label": "Average Fare ($)"}
)
axes[1].set_title("Average Fare ($)\n(Hour × Day of Week)")
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Hour of Day")
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0)

plt.tight_layout()
# plt.savefig("pivot_heatmaps.png", bbox_inches="tight")
plt.show()
print("Plot saved ✅")

# Geographical EDA

In [0]:
import geopandas as gpd

SHAPEFILE_PATH = ("/Volumes/nyc_taxi_analytics/raw/landing_zone/taxi_zones/taxi_zones.shp")

# Load shapefile
gdf = gpd.read_file(SHAPEFILE_PATH)

print(f"Shape         : {gdf.shape}")
print(f"CRS           : {gdf.crs}")
print(f"\nColumns       : {list(gdf.columns)}")
print(f"\nFirst 5 rows  :")
print(gdf.head().to_string())
print(f"\nBorough counts:")
print(gdf["borough"].value_counts())

## Reprojection geospatial data

In [0]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
import numpy as np

# --- Reproject to WGS84 ---
gdf = gdf.to_crs(epsg=4326)
print(f"Reprojected CRS : {gdf.crs}")
print(f"Borough counts  :")
print(gdf["borough"].value_counts())

# --- Aggregate trip data by pickup zone ---
pickup_agg = (df_clean
    .groupby("PULocationID")
    .agg(
        trip_count   = ("fare_amount", "count"),
        avg_fare     = ("fare_amount", "mean"),
        avg_tip      = ("tip_amount",  "mean"),
        total_revenue= ("total_amount","sum"),
        avg_distance = ("trip_distance","mean")
    )
    .reset_index()
    .rename(columns={"PULocationID": "LocationID"})
)

# --- Merge shapefile with trip aggregations ---
gdf_trips = gdf.merge(
    pickup_agg,
    on="LocationID",
    how="left"
)

# Fill zones with no trips as 0
gdf_trips["trip_count"]    = gdf_trips["trip_count"].fillna(0)
gdf_trips["total_revenue"] = gdf_trips["total_revenue"].fillna(0)
gdf_trips["avg_fare"]      = gdf_trips["avg_fare"].fillna(0)

print(f"\nMerged GeoDataFrame shape : {gdf_trips.shape}")
print(f"Zones with trips          : {(gdf_trips['trip_count'] > 0).sum()}")
print(f"Zones with no trips       : {(gdf_trips['trip_count'] == 0).sum()}")

# --- Borough level summary ---
borough_summary = (gdf_trips
    .groupby("borough")
    .agg(
        zones        = ("LocationID",    "count"),
        total_trips  = ("trip_count",    "sum"),
        total_revenue= ("total_revenue", "sum"),
        avg_fare     = ("avg_fare",      "mean")
    )
    .sort_values("total_trips", ascending=False)
    .round(2)
)

print(f"\nBorough Level Summary:")
print("=" * 65)
print(borough_summary.to_string())

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
fig.suptitle("NYC Yellow Taxi 2023 — Zone Level Analysis",
             fontsize=16, fontweight="bold")

# --- Remove EWR for cleaner NYC map ---
gdf_nyc = gdf_trips[gdf_trips["borough"] != "EWR"].copy()

# --- Plot 1: Trip Count Choropleth ---
ax1 = axes[0]
gdf_nyc.plot(
    column       = "trip_count",
    cmap         = "YlOrRd",
    legend       = True,
    legend_kwds  = {"label": "Trip Count", "shrink": 0.6},
    missing_kwds = {"color": "lightgrey", "label": "No trips"},
    linewidth    = 0.3,
    edgecolor    = "white",
    ax           = ax1
)
ax1.set_title("Trip Pickup Volume by Zone", fontsize=13, fontweight="bold")
ax1.set_axis_off()

# Annotate boroughs
borough_centroids = (gdf_nyc
    .dissolve(by="borough")
    .reset_index()
    .to_crs(epsg=4326))
borough_centroids["centroid"] = borough_centroids.geometry.centroid

for _, row in borough_centroids.iterrows():
    ax1.annotate(
        row["borough"],
        xy=(row["centroid"].x, row["centroid"].y),
        fontsize=8, fontweight="bold",
        ha="center", color="black",
        bbox=dict(boxstyle="round,pad=0.2",
                  fc="white", alpha=0.6)
    )

# --- Plot 2: Average Fare Choropleth ---
ax2 = axes[1]
gdf_nyc_fare = gdf_nyc[gdf_nyc["avg_fare"] > 0].copy()
gdf_nyc.plot(
    ax           = ax2,
    color        = "lightgrey",
    linewidth    = 0.3,
    edgecolor    = "white"
)
gdf_nyc_fare.plot(
    column       = "avg_fare",
    cmap         = "Blues",
    legend       = True,
    legend_kwds  = {"label": "Avg Fare ($)", "shrink": 0.6},
    linewidth    = 0.3,
    edgecolor    = "white",
    ax           = ax2
)
ax2.set_title("Average Fare by Pickup Zone", fontsize=13, fontweight="bold")
ax2.set_axis_off()

for _, row in borough_centroids.iterrows():
    ax2.annotate(
        row["borough"],
        xy=(row["centroid"].x, row["centroid"].y),
        fontsize=8, fontweight="bold",
        ha="center", color="black",
        bbox=dict(boxstyle="round,pad=0.2",
                  fc="white", alpha=0.6)
    )

plt.tight_layout()
# plt.savefig("choropleth_maps.png", bbox_inches="tight", dpi=150)
plt.show()
print("Plot saved ✅")

# Insights and Conclusion

## Executive summary

In [0]:
fig = plt.figure(figsize=(20, 14))
fig.suptitle("NYC Yellow Taxi 2023 — Executive Summary Dashboard",
             fontsize=16, fontweight="bold", y=1.01)

# Create grid layout
gs = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])
ax4 = fig.add_subplot(gs[1, 0])
ax5 = fig.add_subplot(gs[1, 1])
ax6 = fig.add_subplot(gs[1, 2])
ax7 = fig.add_subplot(gs[2, :])

day_order = ["Monday","Tuesday","Wednesday",
             "Thursday","Friday","Saturday","Sunday"]

# --- 1. Monthly Volume ---
monthly = (df_clean.groupby("pickup_month")
           .size().reset_index(name="trips"))
monthly["month"] = monthly["pickup_month"].map({
    1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
    7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"})
sns.barplot(x="month", y="trips", data=monthly,
            ax=ax1, palette="Blues_d")
ax1.set_title("Monthly Trip Volume", fontweight="bold")
ax1.set_xlabel("")
ax1.tick_params(axis="x", rotation=45)
ax1.set_ylabel("Trips")

# --- 2. Payment Type ---
payment_map = {1:"Credit Card", 2:"Cash",
               3:"No Charge", 4:"Dispute"}
pay = df_clean["payment_type"].map(payment_map).value_counts()
ax2.pie(pay.values, labels=pay.index,
        autopct="%1.1f%%",
        colors=sns.color_palette("Blues_r", len(pay)),
        startangle=90)
ax2.set_title("Payment Type Split", fontweight="bold")

# --- 3. Time of Day Volume ---
tod_order = ["Morning Rush","Midday",
             "Evening Rush","Night","Late Night"]
tod = (df_clean.groupby("time_of_day")
       .size().reindex(tod_order).reset_index(name="trips"))
sns.barplot(x="time_of_day", y="trips",
            data=tod, ax=ax3, palette="YlOrRd_r")
ax3.set_title("Volume by Time of Day", fontweight="bold")
ax3.set_xlabel("")
ax3.tick_params(axis="x", rotation=30)
ax3.set_ylabel("Trips")

# --- 4. Borough Revenue ---
borough_rev = (gdf_trips.groupby("borough")["total_revenue"]
               .sum()
               .sort_values(ascending=True)
               .drop("EWR"))
borough_rev.plot(kind="barh", ax=ax4,
                 color=sns.color_palette("Blues_d",
                                          len(borough_rev)))
ax4.set_title("Total Revenue by Borough", fontweight="bold")
ax4.set_xlabel("Total Revenue ($)")

# --- 5. Revenue Components ---
rev_components = {
    "Base Fare": df_clean["fare_amount"].mean(),
    "Tip":       df_clean["tip_amount"].mean(),
    "Congestion":df_clean["congestion_surcharge"].mean(),
    "Extra":     df_clean["extra"].mean(),
    "Other":     (df_clean["mta_tax"].mean() +
                  df_clean["improvement_surcharge"].mean() +
                  df_clean["tolls_amount"].mean())
}
rev_s = pd.Series(rev_components)
ax5.pie(rev_s.values, labels=rev_s.index,
        autopct="%1.1f%%",
        colors=sns.color_palette("Blues_r", len(rev_s)),
        startangle=90)
ax5.set_title("Avg Revenue Composition\nper Trip", fontweight="bold")

# --- 6. Fare Efficiency by Rate Code ---
rate_map = {1:"Standard", 2:"JFK", 3:"Newark",
            4:"Nassau/W", 5:"Negotiated", 6:"Group"}
rate_eff = (df_clean.groupby(
                df_clean["RatecodeID"].astype(float)
                .map(rate_map))["fare_per_minute"]
            .mean()
            .sort_values(ascending=True))
rate_eff.plot(kind="barh", ax=ax6,
              color=sns.color_palette("Blues_d",
                                       len(rate_eff)))
ax6.set_title("Avg Fare/Minute\nby Rate Code", fontweight="bold")
ax6.set_xlabel("$/minute")

# --- 7. Daily Time Series ---
daily_ts = (df_clean.groupby("pickup_date")
            .size().reset_index(name="trips"))
daily_ts["pickup_date"] = pd.to_datetime(daily_ts["pickup_date"])
daily_ts["rolling_7d"]  = daily_ts["trips"].rolling(7).mean()
ax7.plot(daily_ts["pickup_date"], daily_ts["trips"],
         color="steelblue", alpha=0.3, linewidth=1)
ax7.plot(daily_ts["pickup_date"], daily_ts["rolling_7d"],
         color="darkblue", linewidth=2.5,
         label="7-day rolling avg")
ax7.set_title("Daily Trip Volume — Full Year 2023",
              fontweight="bold")
ax7.set_xlabel("Date")
ax7.set_ylabel("Daily Trips")
ax7.legend()

plt.savefig("executive_dashboard.png",
            bbox_inches="tight", dpi=150)
plt.show()
print("Dashboard saved ✅")

In [0]:
conclusions = """
╔══════════════════════════════════════════════════════════════════╗
║     NYC YELLOW TAXI 2023 — EDA CONCLUSIONS & RECOMMENDATIONS     ║
╚══════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATASET SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Full 2023 dataset   : ~37.9 million trips
  Stratified sample   : 1,763,368 trips (5% by date-hour)
  Data retained       : 92.98% after cleaning
  Analysis dimensions : Financial, Temporal, Geographical

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEY FINDING 1 — DEMAND CONCENTRATION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  89.8% of all yellow cab trips originate in Manhattan
  Peak demand: Thursday evenings, 16:00–19:00
  Quietest period: All days 03:00–05:00
  Busiest months: May (162K) and October (161K)
  Quietest months: August (130K) and September (129K)

  RECOMMENDATION: Concentrate 90%+ of fleet in Manhattan
  during weekday evening hours. Implement dynamic
  redeployment to outer boroughs only for airport runs.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEY FINDING 2 — REVENUE STRUCTURE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Average revenue per trip : $29.37
  Base fare share          : 66.6% ($19.56)
  Tip share (card only)    : 12.4% ($4.41 true average)
  Regulatory deductions    : 15.1% ($4.43 per trip)
  Cash tip bias            : 17.1% of trips record $0 tip

  RECOMMENDATION: Optimise payment terminal tip presets
  to 25%-28%-30% to leverage default bias ethically.
  Focus tip improvement on Evening Rush — highest
  tip rate at 26.4% — by training drivers on service
  quality during peak demand windows.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEY FINDING 3 — AIRPORT CORRIDOR OPPORTUNITY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Airport trips         : ~8.9% of all trips
  JFK avg fare          : $70.00 flat (3.9% of trips)
  Newark avg fare       : $92.68 (0.3% of trips)
  Newark fare/minute    : $2.46 vs JFK $1.75

  RECOMMENDATION: Newark is significantly underutilised.
  Driver education programme + per-trip Newark bonus
  + return-trip matching system would improve fleet
  economics. JFK volume dominates but Newark delivers
  40% better time efficiency.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEY FINDING 4 — TEMPORAL PATTERNS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Highest fare hour     : 05:00 ($27+ — airport effect)
  Peak volume hour      : 18:00 (125K trips)
  Best revenue/hour     : 16:00–19:00 weekdays
  July 4th              : 60% demand drop — city empties
  Holiday dips          : New Year, Memorial Day,
                          July 4th, Labor Day,
                          Thanksgiving, Christmas

  RECOMMENDATION: Publish annual holiday calendar to
  drivers with advance positioning guidance. Incentivise
  airport-aware drivers for 04:00–06:00 window. Deploy
  maximum fleet Thursday–Friday 16:00–20:00.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KEY FINDING 5 — DATA QUALITY OBSERVATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Schema evolution bug  : airport_fee / Airport_fee
                          split across 2023 — merged
  Vendor data quality   : 3.42% of records from one
                          vendor had systematic nulls
                          and negative values
  Timestamp errors      : 88 records with wrong year
                          (2001–2022, 2024)

  RECOMMENDATION: Implement automated data validation
  at ingestion — flag negative fees, wrong-year
  timestamps, and schema mismatches before data
  reaches analytics layer.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
ANALYTICAL LIMITATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  1. Cash tip data is structurally missing — tip analysis
     applies only to 81.9% credit card transactions
  2. This analysis covers yellow cabs only — green cabs,
     Uber and Lyft are not included
  3. Results based on 5% stratified sample — conclusions
     are representative but not exhaustive
  4. No weather data merged — precipitation effects on
     demand are unquantified

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RECOMMENDED NEXT STEPS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  1. Merge weather data (precipitation, temperature)
     to quantify weather-driven demand spikes
  2. Build demand prediction model using hour,
     day-of-week, month, weather as features
  3. Analyse green cab and FHV data for outer
     borough competitive intelligence
  4. Driver-level analysis — identify top performing
     drivers by fare/minute and model their behaviour
  5. Real-time dashboard for fleet positioning
     using findings from this EDA as business rules
══════════════════════════════════════════════════════════════════
"""

print(conclusions)